# Visual Search System - Evaluation Metrics

This notebook covers the evaluation metrics used to measure the quality of a visual search system. We'll explore both offline metrics (used during model development) and online metrics (used in production).

## Learning Objectives
- Understand ranking metrics for search systems
- Calculate MRR, Precision@k, Recall@k, and mAP
- Master nDCG (Normalized Discounted Cumulative Gain)
- Define online metrics for production monitoring

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import List, Dict, Tuple

## 1. Offline Evaluation Metrics

Offline metrics are computed on a held-out test set before deploying the model. They measure how well the model ranks relevant items.

### Key Concepts
- **Query**: The image the user is searching with
- **Results**: List of images returned, ordered by similarity
- **Relevance**: How similar/relevant each result is to the query
  - Binary: relevant (1) or not relevant (0)
  - Graded: relevance score (e.g., 0-5)

## 2. Mean Reciprocal Rank (MRR)

MRR measures how quickly the **first** relevant result appears.

$$MRR = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{rank_i}$$

Where $rank_i$ is the position of the first relevant result for query $i$.

**Limitation**: Only considers the first relevant result, ignoring others.

In [ ]:
def reciprocal_rank(relevance_list: List[int]) -> float:
    """
    Compute reciprocal rank for a single query.
    
    Args:
        relevance_list: Binary relevance scores for ranked results
    
    Returns:
        Reciprocal rank (1/position of first relevant item)
    """
    for i, rel in enumerate(relevance_list):
        if rel == 1:
            return 1.0 / (i + 1)
    return 0.0

def mean_reciprocal_rank(queries_relevance: List[List[int]]) -> float:
    """
    Compute MRR across multiple queries.
    """
    rr_scores = [reciprocal_rank(rel) for rel in queries_relevance]
    return np.mean(rr_scores)

# Example
query_results = [
    [0, 0, 1, 0, 1],  # First relevant at position 3 → RR = 1/3
    [1, 0, 0, 1, 0],  # First relevant at position 1 → RR = 1/1
    [0, 1, 0, 0, 0],  # First relevant at position 2 → RR = 1/2
]

print("Query Results (1=relevant, 0=not relevant):")
for i, rel in enumerate(query_results):
    rr = reciprocal_rank(rel)
    print(f"  Query {i+1}: {rel} → RR = {rr:.4f}")

mrr = mean_reciprocal_rank(query_results)
print(f"\nMRR = ({1/3:.4f} + {1:.4f} + {1/2:.4f}) / 3 = {mrr:.4f}")

## 3. Precision@k and Recall@k

### Precision@k
Measures what fraction of the top-k results are relevant.

$$Precision@k = \frac{\text{Number of relevant items in top-k}}{k}$$

### Recall@k
Measures what fraction of all relevant items are found in top-k.

$$Recall@k = \frac{\text{Number of relevant items in top-k}}{\text{Total relevant items}}$$

In [ ]:
def precision_at_k(relevance_list: List[int], k: int) -> float:
    """
    Compute Precision@k.
    """
    if k <= 0:
        return 0.0
    top_k = relevance_list[:k]
    return sum(top_k) / k

def recall_at_k(relevance_list: List[int], k: int, total_relevant: int) -> float:
    """
    Compute Recall@k.
    """
    if total_relevant == 0:
        return 0.0
    top_k = relevance_list[:k]
    return sum(top_k) / total_relevant

# Example
relevance = [1, 0, 1, 0, 1, 0, 0, 1, 0, 0]  # 4 relevant items total
total_relevant = sum(relevance)

print("Relevance scores:", relevance)
print(f"Total relevant items: {total_relevant}")
print("\nPrecision@k and Recall@k:")
print("-" * 40)

results = []
for k in [1, 3, 5, 10]:
    p_at_k = precision_at_k(relevance, k)
    r_at_k = recall_at_k(relevance, k, total_relevant)
    results.append({'k': k, 'Precision@k': p_at_k, 'Recall@k': r_at_k})
    print(f"k={k:2d}: Precision@{k} = {p_at_k:.4f}, Recall@{k} = {r_at_k:.4f}")

# Visualize
results_df = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(results_df))
width = 0.35
ax.bar(x - width/2, results_df['Precision@k'], width, label='Precision@k', color='steelblue')
ax.bar(x + width/2, results_df['Recall@k'], width, label='Recall@k', color='darkorange')
ax.set_xlabel('k')
ax.set_ylabel('Score')
ax.set_title('Precision@k vs Recall@k')
ax.set_xticks(x)
ax.set_xticklabels([f'k={k}' for k in results_df['k']])
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### Limitations of Precision@k and Recall@k

1. **Position-agnostic within top-k**: Doesn't distinguish between relevant item at position 1 vs position k
2. **Binary relevance only**: Can't handle graded relevance (e.g., "somewhat relevant" vs "highly relevant")
3. **Fixed k**: Need to choose k in advance

## 4. Average Precision (AP) and Mean Average Precision (mAP)

AP considers the **order** of relevant items by computing precision at each relevant position.

$$AP = \frac{1}{R} \sum_{k=1}^{n} (Precision@k \times rel_k)$$

Where $R$ is the total number of relevant items and $rel_k$ is 1 if item at position $k$ is relevant.

**mAP** is the mean of AP across all queries.

In [ ]:
def average_precision(relevance_list: List[int]) -> float:
    """
    Compute Average Precision for a single query.
    """
    total_relevant = sum(relevance_list)
    if total_relevant == 0:
        return 0.0
    
    precision_sum = 0.0
    relevant_count = 0
    
    for i, rel in enumerate(relevance_list):
        if rel == 1:
            relevant_count += 1
            precision_at_i = relevant_count / (i + 1)
            precision_sum += precision_at_i
    
    return precision_sum / total_relevant

def mean_average_precision(queries_relevance: List[List[int]]) -> float:
    """
    Compute mAP across multiple queries.
    """
    ap_scores = [average_precision(rel) for rel in queries_relevance]
    return np.mean(ap_scores)

# Example: Step-by-step AP calculation
relevance = [1, 0, 1, 0, 1, 0, 0, 1, 0, 0]
print("Step-by-step Average Precision calculation:")
print("Relevance:", relevance)
print("\nPosition | Relevant? | Precision@k | Contribution")
print("-" * 55)

relevant_count = 0
precision_sum = 0.0
for i, rel in enumerate(relevance):
    if rel == 1:
        relevant_count += 1
        p_at_k = relevant_count / (i + 1)
        precision_sum += p_at_k
        print(f"   {i+1:2d}    |    Yes    |   {p_at_k:.4f}    |   {p_at_k:.4f}")
    else:
        print(f"   {i+1:2d}    |    No     |     -        |     -")

ap = precision_sum / sum(relevance)
print(f"\nAP = ({precision_sum:.4f}) / {sum(relevance)} = {ap:.4f}")

## 5. Normalized Discounted Cumulative Gain (nDCG)

nDCG is the **gold standard** for ranking evaluation because it:
1. Handles **graded relevance** (not just binary)
2. Emphasizes **top positions** more heavily (logarithmic discount)
3. **Normalizes** scores to [0, 1] range

### Step 1: Cumulative Gain (CG)
Sum of relevance scores.
$$CG_k = \sum_{i=1}^{k} rel_i$$

### Step 2: Discounted Cumulative Gain (DCG)
Apply logarithmic discount based on position.
$$DCG_k = \sum_{i=1}^{k} \frac{rel_i}{\log_2(i+1)}$$

Or the alternative formula (more common):
$$DCG_k = \sum_{i=1}^{k} \frac{2^{rel_i} - 1}{\log_2(i+1)}$$

### Step 3: Ideal DCG (IDCG)
DCG of the ideal ranking (sort by relevance descending).

### Step 4: Normalized DCG (nDCG)
$$nDCG_k = \frac{DCG_k}{IDCG_k}$$

In [ ]:
def dcg(relevance_scores: List[float], k: int = None) -> float:
    """
    Compute Discounted Cumulative Gain.
    Uses the formula: sum(rel_i / log2(i+1))
    """
    if k is None:
        k = len(relevance_scores)
    
    scores = relevance_scores[:k]
    discounts = np.log2(np.arange(2, len(scores) + 2))
    return np.sum(scores / discounts)

def dcg_exponential(relevance_scores: List[float], k: int = None) -> float:
    """
    Compute DCG with exponential formula.
    Uses the formula: sum((2^rel_i - 1) / log2(i+1))
    """
    if k is None:
        k = len(relevance_scores)
    
    scores = np.array(relevance_scores[:k])
    discounts = np.log2(np.arange(2, len(scores) + 2))
    gains = np.power(2, scores) - 1
    return np.sum(gains / discounts)

def ndcg(relevance_scores: List[float], k: int = None, use_exponential: bool = False) -> float:
    """
    Compute Normalized Discounted Cumulative Gain.
    """
    if k is None:
        k = len(relevance_scores)
    
    dcg_func = dcg_exponential if use_exponential else dcg
    
    # Actual DCG
    actual_dcg = dcg_func(relevance_scores, k)
    
    # Ideal DCG (sort descending)
    ideal_order = sorted(relevance_scores, reverse=True)
    ideal_dcg = dcg_func(ideal_order, k)
    
    if ideal_dcg == 0:
        return 0.0
    
    return actual_dcg / ideal_dcg

# Example with graded relevance
print("nDCG Calculation Example")
print("=" * 60)

# Relevance scores: 0 (not relevant), 1 (somewhat), 2 (relevant), 3 (highly relevant)
relevance = [3, 2, 3, 0, 1, 2]  # Actual ranking
ideal = sorted(relevance, reverse=True)  # [3, 3, 2, 2, 1, 0]

print(f"Actual ranking relevance:  {relevance}")
print(f"Ideal ranking relevance:   {ideal}")
print()

In [ ]:
# Step-by-step DCG calculation
print("Step-by-step DCG Calculation:")
print("-" * 60)
print("Position | Relevance | Discount (log2(i+1)) | Contribution")
print("-" * 60)

total_dcg = 0
for i, rel in enumerate(relevance):
    discount = np.log2(i + 2)
    contribution = rel / discount
    total_dcg += contribution
    print(f"   {i+1}     |     {rel}     |       {discount:.4f}        |    {contribution:.4f}")

print(f"\nDCG = {total_dcg:.4f}")

# IDCG
print("\nIdeal DCG Calculation:")
print("-" * 60)
total_idcg = 0
for i, rel in enumerate(ideal):
    discount = np.log2(i + 2)
    contribution = rel / discount
    total_idcg += contribution
    print(f"   {i+1}     |     {rel}     |       {discount:.4f}        |    {contribution:.4f}")

print(f"\nIDCG = {total_idcg:.4f}")

# nDCG
ndcg_score = total_dcg / total_idcg
print(f"\nnDCG = DCG / IDCG = {total_dcg:.4f} / {total_idcg:.4f} = {ndcg_score:.4f}")

In [ ]:
def visualize_ndcg_concept():
    """Visualize how nDCG penalizes poor rankings"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Three different rankings of the same items
    rankings = [
        {'name': 'Perfect Ranking', 'rel': [3, 3, 2, 2, 1, 0], 'color': 'green'},
        {'name': 'Good Ranking', 'rel': [3, 2, 3, 0, 1, 2], 'color': 'blue'},
        {'name': 'Poor Ranking', 'rel': [0, 1, 2, 2, 3, 3], 'color': 'red'},
    ]
    
    for ax, ranking in zip(axes, rankings):
        rel = ranking['rel']
        positions = range(1, len(rel) + 1)
        
        # Draw bars
        bars = ax.bar(positions, rel, color=ranking['color'], alpha=0.7, edgecolor='black')
        
        # Calculate nDCG
        ndcg_score = ndcg(rel)
        
        ax.set_xlabel('Position')
        ax.set_ylabel('Relevance Score')
        ax.set_title(f"{ranking['name']}\nnDCG = {ndcg_score:.4f}")
        ax.set_ylim(0, 4)
        ax.set_xticks(positions)
    
    plt.suptitle('nDCG Penalizes Poor Rankings\n(Same items, different order)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_ndcg_concept()

### nDCG@k

In practice, we often compute nDCG at specific cutoffs (nDCG@5, nDCG@10, etc.).

In [ ]:
# nDCG at different cutoffs
relevance = [3, 2, 0, 0, 3, 1, 0, 2, 1, 0]

print(f"Relevance scores: {relevance}")
print("\nnDCG at different cutoffs:")
print("-" * 30)

cutoffs = [1, 3, 5, 10]
ndcg_scores = []

for k in cutoffs:
    score = ndcg(relevance, k)
    ndcg_scores.append(score)
    print(f"nDCG@{k:2d} = {score:.4f}")

# Visualize
plt.figure(figsize=(8, 5))
plt.plot(cutoffs, ndcg_scores, 'bo-', markersize=10, linewidth=2)
plt.xlabel('Cutoff (k)')
plt.ylabel('nDCG@k')
plt.title('nDCG Score at Different Cutoffs')
plt.ylim(0, 1.1)
plt.grid(True, alpha=0.3)
for k, score in zip(cutoffs, ndcg_scores):
    plt.annotate(f'{score:.3f}', (k, score), textcoords='offset points',
                xytext=(0, 10), ha='center')
plt.show()

## 6. Metric Comparison

| Metric | Binary Relevance | Graded Relevance | Position-Sensitive | Best For |
|--------|-----------------|------------------|-------------------|----------|
| MRR | ✓ | ✗ | First item only | Finding any relevant item |
| Precision@k | ✓ | ✗ | ✗ (within top-k) | Precision-focused applications |
| Recall@k | ✓ | ✗ | ✗ (within top-k) | Coverage-focused applications |
| mAP | ✓ | ✗ | ✓ | Binary relevance ranking |
| **nDCG** | ✓ | ✓ | ✓ | **General ranking (recommended)** |

## 7. Online Metrics

Online metrics are measured in production from real user behavior. They tell us if the model improvements translate to actual user value.

### 7.1 Click-Through Rate (CTR)

$$CTR = \frac{\text{Number of clicks}}{\text{Number of impressions}}$$

**Considerations:**
- Position bias: Items at top get more clicks regardless of relevance
- Need to normalize by position

In [ ]:
# Simulated online metrics data
np.random.seed(42)

online_data = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=30),
    'impressions': np.random.randint(100000, 150000, 30),
    'clicks': np.random.randint(5000, 15000, 30),
    'avg_time_on_results': np.random.uniform(30, 90, 30),  # seconds
    'searches_per_user': np.random.uniform(2, 5, 30),
})

online_data['ctr'] = online_data['clicks'] / online_data['impressions']

print("Sample Online Metrics:")
print(online_data.head(10).to_string(index=False))

# Visualize CTR trend
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# CTR over time
axes[0, 0].plot(online_data['date'], online_data['ctr'] * 100, 'b-', linewidth=2)
axes[0, 0].fill_between(online_data['date'], online_data['ctr'] * 100, alpha=0.3)
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('CTR (%)')
axes[0, 0].set_title('Click-Through Rate Over Time')
axes[0, 0].grid(True, alpha=0.3)

# Impressions and clicks
axes[0, 1].bar(online_data['date'], online_data['impressions'], alpha=0.5, label='Impressions')
axes[0, 1].bar(online_data['date'], online_data['clicks'], alpha=0.8, label='Clicks')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Impressions vs Clicks')
axes[0, 1].legend()

# Average time on results
axes[1, 0].plot(online_data['date'], online_data['avg_time_on_results'], 'g-', linewidth=2)
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Seconds')
axes[1, 0].set_title('Average Time on Results')
axes[1, 0].grid(True, alpha=0.3)

# Searches per user
axes[1, 1].plot(online_data['date'], online_data['searches_per_user'], 'r-', linewidth=2)
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Searches')
axes[1, 1].set_title('Average Searches per User')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7.2 Key Online Metrics for Visual Search

| Metric | Description | What It Measures |
|--------|-------------|------------------|
| **CTR** | Clicks / Impressions | Result relevance |
| **Time Spent** | Avg time viewing results | Engagement |
| **Searches per Session** | Avg searches per user session | User satisfaction |
| **Refinement Rate** | % searches that lead to refined searches | Result quality (lower is better) |
| **Conversion Rate** | % searches leading to action (save, share, buy) | Business value |

In [ ]:
def online_metrics_dashboard():
    """
    Example of an online metrics monitoring dashboard.
    """
    metrics = {
        'CTR': {
            'current': 8.5,
            'previous': 7.8,
            'unit': '%',
            'direction': 'higher_better'
        },
        'Avg Time on Results': {
            'current': 45.2,
            'previous': 42.1,
            'unit': 'seconds',
            'direction': 'higher_better'
        },
        'Refinement Rate': {
            'current': 12.3,
            'previous': 15.8,
            'unit': '%',
            'direction': 'lower_better'
        },
        'Conversion Rate': {
            'current': 3.2,
            'previous': 2.9,
            'unit': '%',
            'direction': 'higher_better'
        }
    }
    
    print("📊 Visual Search Online Metrics Dashboard")
    print("=" * 60)
    
    for name, data in metrics.items():
        change = data['current'] - data['previous']
        pct_change = (change / data['previous']) * 100
        
        # Determine if change is positive
        if data['direction'] == 'higher_better':
            is_positive = change > 0
        else:
            is_positive = change < 0
        
        arrow = '↑' if change > 0 else '↓'
        status = '✅' if is_positive else '⚠️'
        
        print(f"\n{name}:")
        print(f"  Current: {data['current']}{data['unit']}")
        print(f"  Change: {arrow} {abs(change):.1f}{data['unit']} ({pct_change:+.1f}%) {status}")

online_metrics_dashboard()

## 8. A/B Testing for Visual Search

When deploying a new model, use A/B testing to compare against the current production model.

In [ ]:
from scipy import stats

def ab_test_analysis(control_clicks, control_impressions, 
                     treatment_clicks, treatment_impressions,
                     confidence_level=0.95):
    """
    Analyze A/B test results for CTR.
    """
    # Calculate CTRs
    control_ctr = control_clicks / control_impressions
    treatment_ctr = treatment_clicks / treatment_impressions
    
    # Pooled proportion for z-test
    pooled = (control_clicks + treatment_clicks) / (control_impressions + treatment_impressions)
    
    # Standard error
    se = np.sqrt(pooled * (1 - pooled) * (1/control_impressions + 1/treatment_impressions))
    
    # Z-statistic
    z_stat = (treatment_ctr - control_ctr) / se
    
    # P-value (two-tailed)
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    
    # Confidence interval
    z_critical = stats.norm.ppf(1 - (1 - confidence_level) / 2)
    diff = treatment_ctr - control_ctr
    ci_lower = diff - z_critical * se
    ci_upper = diff + z_critical * se
    
    return {
        'control_ctr': control_ctr,
        'treatment_ctr': treatment_ctr,
        'absolute_diff': diff,
        'relative_diff': (diff / control_ctr) * 100,
        'p_value': p_value,
        'significant': p_value < (1 - confidence_level),
        'ci_lower': ci_lower,
        'ci_upper': ci_upper
    }

# Example A/B test
results = ab_test_analysis(
    control_clicks=8000, control_impressions=100000,
    treatment_clicks=8800, treatment_impressions=100000
)

print("A/B Test Results: New Embedding Model")
print("=" * 50)
print(f"\nControl (Current Model):")
print(f"  CTR: {results['control_ctr']*100:.2f}%")
print(f"\nTreatment (New Model):")
print(f"  CTR: {results['treatment_ctr']*100:.2f}%")
print(f"\nDifference:")
print(f"  Absolute: {results['absolute_diff']*100:+.2f}%")
print(f"  Relative: {results['relative_diff']:+.1f}%")
print(f"\nStatistical Significance:")
print(f"  P-value: {results['p_value']:.4f}")
print(f"  Significant at 95%: {'Yes ✅' if results['significant'] else 'No ❌'}")
print(f"  95% CI: [{results['ci_lower']*100:.2f}%, {results['ci_upper']*100:.2f}%]")

## 9. Summary

### Key Takeaways

1. **Offline Metrics for Development:**
   - Use **nDCG** as the primary metric (handles graded relevance)
   - Track Precision@k and Recall@k for specific use cases
   - mAP for binary relevance scenarios

2. **Online Metrics for Production:**
   - CTR measures user engagement with results
   - Time spent indicates content quality
   - Refinement rate shows search satisfaction

3. **A/B Testing:**
   - Always validate offline improvements with online experiments
   - Use statistical significance testing
   - Consider multiple metrics together

In [ ]:
def metrics_summary():
    summary = """
    ╔══════════════════════════════════════════════════════════════╗
    ║         VISUAL SEARCH EVALUATION METRICS SUMMARY             ║
    ╠══════════════════════════════════════════════════════════════╣
    ║                                                              ║
    ║   OFFLINE METRICS (Development):                             ║
    ║   ├── nDCG@k: Primary metric (graded, position-aware)       ║
    ║   ├── Precision@k: Fraction of relevant in top-k            ║
    ║   ├── Recall@k: Coverage of relevant items                  ║
    ║   └── mAP: Average precision across queries                 ║
    ║                                                              ║
    ║   ONLINE METRICS (Production):                               ║
    ║   ├── CTR: Click-through rate                               ║
    ║   ├── Time Spent: Engagement with results                   ║
    ║   ├── Refinement Rate: Search satisfaction                  ║
    ║   └── Conversion Rate: Business value                       ║
    ║                                                              ║
    ║   BEST PRACTICES:                                            ║
    ║   • Use nDCG as primary offline metric                      ║
    ║   • Validate with A/B tests before full deployment          ║
    ║   • Monitor multiple online metrics together                ║
    ║   • Account for position bias in click data                 ║
    ║                                                              ║
    ╚══════════════════════════════════════════════════════════════╝
    """
    print(summary)

metrics_summary()